# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

- **Title:** Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya
- **Identifier:** 10.71728/senscience.y7m0-f273
- **URL:** https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
# Access the dataset metadata (as a Python object)
metadata = dataset.metadata
print(f"Dataset: {metadata.name}\n\n{metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s (unique identifiers in the Croissant schema).

*All entities are referenced using their `@id`. For full schema exploration, use the methods on the Dataset object.*

In [ ]:
# List all record sets available in this dataset, using their @id
record_sets = [rs for rs in dataset.record_sets]
if not record_sets:
    print("No record sets detected in the Croissant schema. Check the distribution info or schema for tabular data components.")
else:
    print("Available record sets:")
    for rs in record_sets:
        print(f"\nRecordSet Name: {rs.name}\n  @id: {rs.id}\n  Description: {getattr(rs, 'description', '(no description)')}")
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {field.name} (@id: {field.id}, type: {getattr(field, 'data_type', 'unknown')})")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames. Each is keyed by its `@id` for consistency, and columns are referenced by their field `@id`s.

In [ ]:
# Extract data from ALL available record sets
record_set_ids = [rs.id for rs in dataset.record_sets]

dataframes = {}  # Dict of record_set_id -> pd.DataFrame
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set '{record_set_id}'.")

if record_set_ids:
    primary_rs_id = record_set_ids[0]
    print(f"\nExample columns for record set: {primary_rs_id}")
    print(dataframes[primary_rs_id].columns.tolist())
    dataframes[primary_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply typical data filtering, normalization, and summarization. We demonstrate basic EDA on a numeric field, referencing by its canonical `@id`.

In [ ]:
# EDA - Demonstrate on the first available numeric field for the first record set
import numpy as np

if not record_set_ids:
    print("No record sets available for EDA.")
else:
    rs = [r for r in dataset.record_sets if r.id == primary_rs_id][0]
    df = dataframes[primary_rs_id]
    
    # Find a numeric field (`data_type` in ['Integer', 'Float', 'Number'])
    numeric_fields = [f for f in getattr(rs, 'fields', []) if getattr(f, 'data_type', None) in ['Integer', 'Float', 'Number']]
    if not numeric_fields or df.empty:
        print("No numeric fields found for this record set.")
    else:
        # Use the first numeric field found
        numeric_field = numeric_fields[0]
        numeric_field_id = numeric_field.id
        print(f"Using numeric field: {numeric_field.name} (@id: {numeric_field_id})")

        # Convert the column to numeric, handle errors
        numeric_series = pd.to_numeric(df[numeric_field_id], errors='coerce')
        threshold = numeric_series.mean() if ~numeric_series.isna().all() else 10
        filtered_df = df[numeric_series > threshold].copy()
        print(f"Filtered records where {numeric_field_id} > {threshold:.2f} ({len(filtered_df)} records):")
        print(filtered_df[[numeric_field_id]].head())

        # Normalization example
        filtered_df[f"{numeric_field_id}_normalized"] = (numeric_series - numeric_series.mean())/numeric_series.std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a string/categorical field for grouping
        group_fields = [f for f in rs.fields if getattr(f, 'data_type', None) in ['Text', 'String']]
        if group_fields:
            group_field = group_fields[0].id
            if group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().to_frame()
                print(f"\nGrouped data by {group_field} (mean of {numeric_field_id}):")
                print(grouped_df.head())
        else:
            print("No text/categorical field available for grouping.")

## 5. Visualization
Visualize numeric field distributions or relationships, using pandas and matplotlib.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if not record_set_ids or not numeric_fields or df.empty:
    print("No data to visualize.")
else:
    # Histogram for the numeric field
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(df[numeric_field_id], errors='coerce').dropna(), bins=20)
    plt.title(f"Distribution of {numeric_field.name} (@id: {numeric_field_id})")
    plt.xlabel(numeric_field.name)
    plt.ylabel("Frequency")
    plt.show()

    # If grouping field exists, show boxplot
    if group_fields and (group_field in df.columns):
        plt.figure(figsize=(10,5))
        sns.boxplot(x=df[group_field], y=pd.to_numeric(df[numeric_field_id], errors='coerce'))
        plt.title(f"{numeric_field.name} by {group_fields[0].name}")
        plt.xlabel(group_fields[0].name)
        plt.ylabel(numeric_field.name)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
In this notebook, you have:
- Loaded a FAIR^2 dataset from a Croissant schema.
- Identified available record sets and fields using their `@id`s.
- Extracted records to pandas DataFrames.
- Performed basic EDA, including filtering and normalizing numeric fields by `@id`.
- Visualized data distributions.

**Next steps:** Tailor analysis to your research question(s), and always reference entities by their Croissant `@id`, ensuring reproducibility and semantic clarity for downstream processing.